# 02 — Spread Analysis

Focus pair: **ERCOT vs PJM**

Why this pair is interesting:
- Different weather exposure (Gulf Coast heat vs Mid-Atlantic seasons)
- Different generation mix (ERCOT: gas/wind, PJM: gas/coal/nuclear)
- ERCOT is electrically isolated — no physical interconnection to PJM
- High volatility difference creates spread trading opportunities

We test for **cointegration** (is the spread stationary?) and estimate the
**half-life of mean reversion** (how fast does it snap back?).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import ISODataFetcher
from src.analysis.spreads import SpreadAnalyzer

In [ ]:
fetcher = ISODataFetcher(config_path='../config/settings.yaml')
analyzer = SpreadAnalyzer()

end_date = pd.Timestamp.now().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.now() - pd.Timedelta(days=365)).strftime('%Y-%m-%d')

ercot = fetcher.fetch('ERCOT', start_date, end_date)
pjm = fetcher.fetch('PJM', start_date, end_date)

spread_df = analyzer.compute_spread(ercot, pjm)
print(f"Spread series: {len(spread_df)} days")
spread_df.head()

In [ ]:
# Plot the spread
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['ERCOT vs PJM Daily Prices',
                                    'Spread (ERCOT - PJM)',
                                    'Z-Score (20-day rolling)'])

fig.add_trace(go.Scatter(x=spread_df['trade_date'], y=spread_df['price_a'],
                         name='ERCOT', line=dict(color='red')), row=1, col=1)
fig.add_trace(go.Scatter(x=spread_df['trade_date'], y=spread_df['price_b'],
                         name='PJM', line=dict(color='blue')), row=1, col=1)

fig.add_trace(go.Scatter(x=spread_df['trade_date'], y=spread_df['spread'],
                         name='Spread', line=dict(color='green')), row=2, col=1)
fig.add_hline(y=spread_df['spread'].mean(), line_dash='dash', row=2, col=1)

zscore = analyzer.rolling_zscore(spread_df['spread'], window=20)
fig.add_trace(go.Scatter(x=spread_df['trade_date'], y=zscore,
                         name='Z-Score', line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=1.5, line_dash='dot', line_color='red', row=3, col=1)
fig.add_hline(y=-1.5, line_dash='dot', line_color='red', row=3, col=1)
fig.add_hline(y=0, line_dash='dash', row=3, col=1)

fig.update_layout(height=800, title_text='ERCOT-PJM Spread Analysis')
fig.show()

In [ ]:
# Cointegration test
coint_result = analyzer.cointegration_test(
    spread_df['price_a'].values,
    spread_df['price_b'].values
)
print("Cointegration Test (Engle-Granger):")
for k, v in coint_result.items():
    print(f"  {k}: {v}")

In [ ]:
# Half-life of mean reversion
hl = analyzer.half_life(spread_df['spread'])
print(f"Half-life of mean reversion: {hl:.1f} days")
print(f"Hurst exponent: {analyzer.hurst_exponent(spread_df['spread']):.3f}")

# ADF test on the spread
adf = analyzer.adf_test(spread_df['spread'])
print(f"\nADF test on spread:")
print(f"  Stationary: {adf['stationary']} (p={adf['p_value']:.4f})")

In [ ]:
# Full spread summary
summary = analyzer.spread_summary(spread_df['spread'])
print("\nSpread Summary:")
for k, v in summary.items():
    if isinstance(v, dict):
        print(f"  {k}:")
        for kk, vv in v.items():
            print(f"    {kk}: {vv}")
    else:
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Interpretation

- If **cointegrated** (p < 0.05): the spread is stationary and suitable for mean-reversion trading
- **Half-life** tells us how many days it takes for a spread deviation to decay by 50%
- **Hurst < 0.5**: confirms mean-reverting behavior
- The z-score chart shows entry/exit opportunities: trade when |z| > 1.5, exit when |z| < 0.3